# Train, Valid, Test Split (ml.m5.12xlarge)

In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

### Functions

In [2]:
# denote df
def mark_df(flt_prop_row):
    if flt_prop_row <= 0.6:
        return 'train'
    elif flt_prop_row <= 0.8:
        return 'valid'
    else:
        return 'test'

### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')
str_dirname_output = './output'

str_id = 'uniqueid'
str_datecol = 'applicationdate__app'
str_target = 'target'

list_cols_id = [
    str_id,
    str_datecol,
    str_target,
]

flt_mean_desired = 0.653

Project: 20231010-gen-xii
Step: 03_train_valid_test_split


### Output

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Import data

In [5]:
%%time

# import
str_filename = 'df_raw.gzip'
str_uri = f's3://{str_project}/03_pricing_lgd/01_data_prep/01_data_collection/output/{str_filename}'
# read from s3
df = pd.read_parquet(str_uri)
# replace
df.replace(['NaN','nan'], np.nan, inplace=True)

# show
df

<timed exec>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


CPU times: user 2.17 s, sys: 975 ms, total: 3.15 s
Wall time: 2.25 s


,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,RunningNetLoss,bitTarget24Months,target
11,20181120.0,20181120.0,NaN,201810.0,106347.0,2018-11-20,1.0,AU,2018-11-29,NaN,...,0.330848,0.107492,1,1.084734,suv,1,2012-02-10 13:48:47.720,20387.06,1,0.985462
12,20181120.0,20181120.0,NaN,201810.0,106347.0,2018-11-20,1.0,AU,2018-11-29,NaN,...,0.330848,0.107492,1,1.084734,suv,1,2012-02-10 13:48:47.720,20387.06,1,0.985462
19,20181017.0,20181017.0,NaN,201810.0,39394.0,2018-10-17,1.0,AU,2018-10-17,6934.0,...,0.814385,0.341774,1,1.005341,auto,0,2006-04-16 10:50:55.000,4442.31,1,0.272984
20,20181017.0,20181017.0,NaN,201810.0,39394.0,2018-10-17,1.0,AU,2018-10-17,6934.0,...,0.814385,0.341774,1,1.005341,auto,0,2006-04-16 10:50:55.000,4442.31,1,0.272984
21,20181017.0,20181017.0,NaN,201810.0,34747.0,2018-10-17,1.0,AU,2018-10-17,17428.0,...,0.380302,0.149874,0,1.253794,auto,0,2003-07-08 14:32:29.097,9584.90,1,0.434160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.203666,0.100599,0,1.210315,truck,0,2013-04-02 15:33:30.193,28198.75,1,0.939958
186165,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.358319,0.113501,1,1.176435,auto,1,2014-09-09 16:07:25.123,13195.01,1,0.754292
186166,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.334282,0.121807,0,1.150000,auto,1,2012-11-05 16:48:22.563,7251.90,1,0.543776
186168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.419962,0.109622,0,1.249284,auto,0,2011-06-21 16:14:20.870,11811.67,1,0.981076


### Mark each row as train, valid, or test

In [6]:
%%time

# sort
df.sort_values(by=str_datecol, ascending=True, inplace=True)

# create column
int_nrows = df.shape[0]
df['int_row'] = [i for i in range(1, int_nrows + 1)]
# divide by n rows
df['flt_prop_row'] = df['int_row'] / int_nrows

# drop
df.drop('int_row', axis=1, inplace=True)

# mark the df
df['data_set'] = df['flt_prop_row'].apply(mark_df)

# show
df

CPU times: user 149 ms, sys: 208 ms, total: 357 ms
Wall time: 366 ms


,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,RunningNetLoss,bitTarget24Months,target,flt_prop_row,data_set
38325,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1.106635,auto,0,2012-06-18 09:31:54.393,9070.52,1,0.624137,0.000049,train
38324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1.106635,auto,0,2012-06-18 09:31:54.393,9070.52,1,0.624137,0.000098,train
38328,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1.249982,auto,0,2009-10-15 16:06:40.767,10250.98,1,0.543609,0.000146,train
38329,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1.249982,auto,0,2009-10-15 16:06:40.767,10250.98,1,0.543609,0.000195,train
38371,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1.160797,auto,0,2008-02-01 13:09:22.957,927.71,1,0.048716,0.000244,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21912,20191228.0,20191228.0,NaN,201910.0,860743.0,2019-12-28,1.0,AU,2019-12-29,10206.0,...,1,1.136686,auto,0,2011-03-22 08:47:41.800,6247.09,1,0.332609,0.999805,test
25854,20191230.0,20191230.0,NaN,201910.0,920763.0,2019-12-30,1.0,AU,2019-12-30,13472.0,...,0,1.074224,suv,0,2010-07-30 13:50:37.207,13097.63,1,0.459316,0.999854,test
33982,20191230.0,20191230.0,NaN,201910.0,957548.0,2019-12-30,1.0,AU,2019-12-31,10545.0,...,0,1.200000,auto,1,2015-09-18 16:11:35.860,7723.58,1,0.713294,0.999902,test
1570,20191231.0,20191231.0,NaN,201910.0,972068.0,2019-12-31,1.0,AU,2019-12-31,9408.0,...,0,1.149621,auto,0,2009-06-11 17:08:59.937,7835.19,1,0.474860,0.999951,test


### Show value counts

In [7]:
ser_prop = pd.value_counts(df['data_set'], normalize=True)
print(ser_prop)

data_set
train    0.599961
valid    0.200020
test     0.200020
Name: proportion, dtype: float64


/tmp/ipykernel_23902/1628931787.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  ser_prop = pd.value_counts(df['data_set'], normalize=True)


In [8]:
ser_freq = pd.value_counts(df['data_set'], normalize=False)
print(ser_freq)

data_set
train    12298
valid     4100
test      4100
Name: count, dtype: int64


/tmp/ipykernel_23902/3389291447.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  ser_freq = pd.value_counts(df['data_set'], normalize=False)


### Write data frames

In [9]:
list_df = [
    'train',
    'valid',
    'test',
]
for str_df in tqdm (list_df):
    # subset
    df_tmp = df[df['data_set']==str_df].copy()
    
    # logic to get factor
    if str_df == 'train':
        # iterate
        list_flt_factor = list(np.linspace(-1, 1, 1000))
        for flt_factor in list_flt_factor:
            df_tmp['target_tmp'] = df_tmp['target'] + flt_factor
            flt_mean_tmp = df_tmp['target_tmp'].mean()
            if flt_mean_tmp >= flt_mean_desired:
                df_tmp.drop('target_tmp', axis=1, inplace=True)
                break
            else:
                pass
    else:
        pass
    
    # apply factor
    df_tmp['target'] = df_tmp['target'] + flt_factor
    
    # drop
    list_cols = [
        'flt_prop_row',
        'data_set',
    ]
    df_tmp.drop(list_cols, axis=1, inplace=True)
    
    # write to s3
    str_filename = f'df_{str_df}_raw.gzip'
    str_uri = f's3://{str_project}/03_pricing_lgd/01_data_prep/{str_step}/{str_filename}'
    df_tmp.to_parquet(str_uri, compression='gzip')

100%|██████████| 3/3 [00:09<00:00,  3.16s/it]
